# Patient No-Show Prediction

---

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Load dataset
df = pd.read_csv("no_show_dataset.csv")

# Inspect data
display(df.head())
display(df.info())
display(df['No-show'].value_counts())

## 2. Data Cleaning and Feature Engineering

--- 

### 2.1 Handle Invalid Data

In [ ]:
# Remove invalid ages
df = df[df['Age'] >= 0]

# Convert to datetime
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# Feature: waiting days
df['WaitingDays'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days

# Day of week and month of appointment
df['AppointmentDayOfWeek'] = df['AppointmentDay'].dt.dayofweek
df['AppointmentMonth'] = df['AppointmentDay'].dt.month

# Gender: F=0, M=1
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})

# One-hot encode neighbourhoods
df = pd.get_dummies(df, columns=['Neighbourhood'], drop_first=True)

# Target: No-show Yes=1, No=0
df['No-show'] = df['No-show'].map({'Yes': 1, 'No': 0})

# Convert condition columns to numeric 0/1
conditions = ['Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received']
for col in conditions:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].map({'0': 0, '1': 1, 'No': 0, 'Yes': 1}).fillna(0).astype(int)

# Drop unnecessary columns
df = df.drop(columns=['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay'])

## 3. Exploratory Data Analysis (EDA)

---

### 3.1 Target Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='No-show', data=df)
plt.title('No-show Distribution')
plt.savefig('plots/no_show_distribution.png')
plt.show()

print("No-show rate:\n", df['No-show'].value_counts(normalize=True))

### 3.2 Age vs No-show

In [ ]:
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,18,30,45,60,100])
plt.figure(figsize=(8,5))
sns.barplot(x='AgeGroup', y='No-show', data=df, ci=None)
plt.title('No-show rate by Age Group')
plt.savefig('plots/age_vs_no_show.png')
plt.show()

### 3.3 WaitingDays vs No-show

In [ ]:
df['WaitingGroup'] = pd.cut(df['WaitingDays'], bins=[-1,0,5,10,20,50,100,300])
plt.figure(figsize=(8,5))
sns.barplot(x='WaitingGroup', y='No-show', data=df, ci=None)
plt.title('No-show rate by Waiting Days')
plt.savefig('plots/waiting_vs_no_show.png')
plt.show()

### 3.4 Gender vs No-show

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Gender', hue='No-show', data=df)
plt.title('Gender vs No-show')
plt.savefig('plots/gender_vs_no_show.png')
plt.show()

### 3.5 SMS vs No-show

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='SMS_received', hue='No-show', data=df)
plt.title('SMS Received vs No-show')
plt.savefig('plots/sms_vs_no_show.png')
plt.show()

### 3.6 Medical Conditions vs No-show

In [ ]:
for col in conditions:
    rate = df.groupby(col)['No-show'].mean()
    print(f'{col} no-show rate:\n{rate}\n')

### 3.7 Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,8))

# Select only numeric columns
numeric_cols = df.select_dtypes(include='number')
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f')
plt.title('Feature Correlation')
plt.savefig('plots/feature_correlation.png')
plt.show()


## 4. Model Preparation


In [ ]:
# Drop binned categorical columns from features
X = df.drop(columns=['No-show', 'AgeGroup', 'WaitingGroup'])
y = df['No-show']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 5. Baseline Model: Logistic Regression


In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:,1]

print("Logistic Regression Metrics:\n")
print(classification_report(y_test, y_pred_lr))
print("AUC:", roc_auc_score(y_test, y_proba_lr))

## 6. Advanced Model: Random Forest


In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print("Random Forest Metrics:\n")
print(classification_report(y_test, y_pred_rf))
print("AUC:", roc_auc_score(y_test, y_proba_rf))

## 6.1 Feature Importance

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8,6))
feat_imp[:10].plot(kind='barh')
plt.title('Top 10 Features by Importance')
plt.savefig('plots/top10_feature_importance.png')
plt.show()

## 7. Insights & Conclusion

### 7.1 Target Distribution
- Overall no-show rate: ~20%
- Most patients attend their appointments.

### 7.2 Key Observations from EDA
- **Age:** Younger patients more likely to miss appointments (18-30: 24.7%, 60-100: 15.2%)
- **Waiting Time:** Longer waiting increases no-show (20-50 days: 33.5%)
- **Gender:** Little difference (Female 20.3%, Male 19.9%)
- **SMS Reminder:** SMS sent correlates with higher no-show (SMS 27.5%, No SMS 16.7%)
- **Medical Conditions:** Scholarship and some conditions slightly increase no-show

### 7.3 Model Performance
| Model                 | AUC  | F1-score |
|-----------------------|------|----------|
| Logistic Regression   | 0.78 | 0.62     |
| Random Forest         | 0.81 | 0.68     |

### 7.4 Top Features (Random Forest)
- WaitingDays
- SMS_received
- Age
- AppointmentDayOfWeek
- Scholarship
- Diabetes
- Hipertension
- Handcap

### 7.5 CV-Ready Bullets
- Built ML model predicting patient no-shows, achieving AUC 0.81.
- Identified key predictors: waiting days, SMS reminders, age, scholarships.
- Conducted feature engineering, EDA, and model evaluation (Logistic Regression & Random Forest).
- Presented findings with clear visualisations and actionable insights for hospital operations.
